In [27]:
import pandas as pd
import requests
import json
import pandas as pd
from tqdm import tqdm

In [28]:
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
MODEL_NAME = "llama3.2:3b"

In [29]:

# df = pd.read_csv("mozilla_core_clean.csv")
# Rename by specific mapping
# df = df.rename(columns={"Unnamed: 0": "id"})
# eval_docs = df.sample(400, random_state=42)

In [30]:
# eval_docs.head()

In [31]:
# eval_docs.to_csv("random_samples.csv", index=False)

In [32]:
eval_docs = pd.read_csv("random_samples.csv")

In [33]:
def generate_queries(bug_text):
    prompt = f"""
You are generating evaluation queries for a retrieval system.

Given the following bug report, generate EXACTLY 2 realistic developer search queries.

Return output strictly in JSON format like this:
{{
  "queries": [
    "query 1",
    "query 2"
  ]
}}

BUG REPORT:
\"\"\"{bug_text}\"\"\"
"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3
        }
    }

    response = requests.post(OLLAMA_URL, json=payload)
    result = response.json()["response"]
    # result = None
    # print(response.status_code)
    # print(response.text)
    # Extract JSON safely
    try:
        json_start = result.find("{")
        json_end = result.rfind("}") + 1
        parsed = json.loads(result[json_start:json_end])
        return parsed["queries"]
    except:
        return []

In [34]:
evaluation_data = []

for _, row in tqdm(eval_docs.iterrows(), total=len(eval_docs)):
    bug_text = row["Description"]
    doc_id = row["Id"]

    queries = generate_queries(bug_text)

    for q in queries:
        evaluation_data.append({
            "query": q,
            "relevant_doc_id": doc_id
        })

eval_df = pd.DataFrame(evaluation_data)
eval_df.to_csv("evaluation_queries.csv", index=False)

print("Saved evaluation_queries.csv")

100%|██████████| 400/400 [09:07<00:00,  1.37s/it]

Saved evaluation_queries.csv
